In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Model ve tokenizer-HuggingFace üzerinden kullanılacak Türkçe BERT modelinin adı.
model_name = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

Downloading:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

c:\ProgramData\anaconda3\lib\site-packages\huggingface_hub\file_download.py:123: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ertuğrul\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Downloading:   0%|          | 0.00/385 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/251k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-base-turkish-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Bu fonksiyon, bir cümle içindeki belirli bir kelimenin (örneğin "yüz"), BERT modeliyle bağlama duyarlı embedding’ini (vektör temsilini) çıkartıyor.

In [ ]:
def get_word_embedding(sentence, target_word):
    encoded = tokenizer(sentence, return_tensors="pt", return_offsets_mapping=True)
    offset_mapping = encoded.pop("offset_mapping") 
    outputs = model(**encoded)

    tokens = tokenizer.tokenize(sentence)
    last_hidden = outputs.last_hidden_state.squeeze(0)

    # Hedef kelimenin geçtiği ilk token'i bul
    for idx, token in enumerate(tokens):
        if target_word.lower() in token.lower():
            return last_hidden[idx].detach().numpy()

    return None

In [6]:
# Örnek cümleler
c1 = "Yüzünü pencereden dışarı çıkardı ve derin bir nefes aldı."
c2 = "Denizde kulaç atarak saatlerce yüzdü."

# Embedding'leri al
e1 = get_word_embedding(c1, "yüz")
e2 = get_word_embedding(c2, "yüz")



In [7]:
# Benzerliği ölç
similarity = cosine_similarity([e1], [e2])[0][0]
print(f"Kosinüs benzerliği: {similarity:.3f}")

Kosinüs benzerliği: 0.039


In [10]:
c3 = "Sabah erkenden denize girip uzun süre yüzdü."
c4 = "Her hafta havuzda düzenli olarak yüzüyor."

e3 = get_word_embedding(c3, "yüz")
e4 = get_word_embedding(c4, "yüz")

similarity_same_meaning = cosine_similarity([e3], [e4])[0][0]
print(f"Aynı anlamlı cümlelerde 'yüz' benzerliği: {similarity_same_meaning:.3f}")

Aynı anlamlı cümlelerde 'yüz' benzerliği: 0.603
